In [ ]:
import cv2
import torch
import numpy as np
import time
import os
from ultralytics import YOLO
from torchvision import transforms
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

class CSRNet(torch.nn.Module):
    def __init__(self, load_weights=True):
        super(CSRNet, self).__init__()
        self.frontend_feat = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512]
        self.backend_feat = [512, 512, 512, 256, 128, 64]
        self.frontend = make_layers(self.frontend_feat)
        self.backend = make_layers(self.backend_feat, in_channels=512, dilation=True)
        self.output_layer = torch.nn.Conv2d(64, 1, kernel_size=1)
        if load_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x

    def _initialize_weights(self):
        print("Initialized CSRNet weights (using VGG16 base).")

def make_layers(cfg, in_channels=3, batch_norm=False, dilation=False):
    d_rate = 2 if dilation else 1
    layers = []
    for v in cfg:
        if v == 'M':
            layers += [torch.nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = torch.nn.Conv2d(in_channels, v, kernel_size=3, padding=d_rate, dilation=d_rate)
            layers += [conv2d, torch.nn.ReLU(inplace=True)]
            in_channels = v
    return torch.nn.Sequential(*layers)


YOLO_PERSON_MODEL_PATH = 'yolov8n.pt' 
person_detector = YOLO(YOLO_PERSON_MODEL_PATH)
print("YOLOv8 Person Detector loaded.")

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print("Haar Cascade Face Detector loaded (but using YOLO head masking for privacy).") 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSRNet_WEIGHTS_PATH = 'weights.pth' 
csr_net = CSRNet(load_weights=False) 

CSRNet_LOADED_SUCCESSFULLY = False 
try:
 
    csr_net.to(device).eval()
    CSRNet_LOADED_SUCCESSFULLY = True
    print(f"CSRNet Model initialized on {device}.")
except Exception as e:
    print(f"Error loading CSRNet weights: {e}. Running with DUMMY CSRNet (count=0).")

csr_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

COUNT_HISTORY = []
WINDOW_SIZE = 5

def fuse_and_smooth_counts(yolo_count, csr_count, csr_loaded):
    if csr_loaded:
        current_count = csr_count
    else:
        current_count = yolo_count 

    global COUNT_HISTORY
    COUNT_HISTORY.append(current_count)
    if len(COUNT_HISTORY) > WINDOW_SIZE:
        COUNT_HISTORY.pop(0)
    
    smoothed_count = int(np.mean(COUNT_HISTORY)) if COUNT_HISTORY else current_count
        
    return smoothed_count


def anonymize_head_blur(frame, person_results):
    anonymized_frame = frame.copy()
    
    faces_count = 0 

    if len(person_results) > 0 and person_results[0].boxes:
        for box in person_results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            
            head_height = (y2 - y1) // 3 
            head_y1 = y1
            head_y2 = y1 + head_height
            
            head_y1 = max(0, head_y1)
            head_y2 = min(anonymized_frame.shape[0], head_y2)

            if head_y2 > head_y1:
                head_roi = anonymized_frame[head_y1:head_y2, x1:x2]
                blurred_head = cv2.GaussianBlur(head_roi, (51, 51), 0) 
                anonymized_frame[head_y1:head_y2, x1:x2] = blurred_head
                faces_count += 1 
    
    return anonymized_frame, faces_count


def estimate_crowd_count(frame, csr_net, csr_transform, device, is_loaded):
    
    if not is_loaded:
        return 0, np.zeros((frame.shape[0], frame.shape[1]))

    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = csr_transform(img_pil).to(device)
    
    with torch.no_grad():
        output = csr_net(img_tensor.unsqueeze(0)) 

    predicted_count = int(output.sum().item())
    
    density_map = output.squeeze(0).squeeze(0).cpu().numpy()
    density_map_resized = cv2.resize(density_map, (frame.shape[1], frame.shape[0]))
    
    return predicted_count, density_map_resized


def process_frame_pipeline(frame, person_detector, face_cascade, csr_net, csr_transform, device, csr_loaded):
    
    person_results = person_detector.predict(frame.copy(), classes=[0], conf=0.4, verbose=False) 
    yolo_count = 0
    if len(person_results) > 0 and person_results[0].boxes:
        yolo_count = len(person_results[0].boxes)

    anonymized_frame, masked_heads_count = anonymize_head_blur(frame, person_results)

    csr_count, density_map_resized = estimate_crowd_count(anonymized_frame, csr_net, csr_transform, device, csr_loaded)
    
    final_smoothed_count = fuse_and_smooth_counts(yolo_count, csr_count, csr_loaded)

    if len(person_results) > 0 and person_results[0].boxes:
        for box in person_results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(anonymized_frame, (x1, y1), (x2, y2), (0, 255, 0), 2) 
    
    if csr_loaded and np.max(density_map_resized) > 0:
        heatmap_img = np.uint8(255 * density_map_resized / np.max(density_map_resized))
        heatmap_colored = cv2.applyColorMap(heatmap_img, cv2.COLORMAP_JET)
        alpha = 0.4  
        cv2.addWeighted(heatmap_colored, alpha, anonymized_frame, 1 - alpha, 0, anonymized_frame)

    cv2.putText(anonymized_frame, f"CSRNet Count: {csr_count} ({'DENSE' if csr_loaded else 'DUMMY'})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(anonymized_frame, f"YOLO Count (Sparse): {yolo_count}", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(anonymized_frame, f"Final Smoothed Count: {final_smoothed_count}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2, cv2.LINE_AA) 
    
    return anonymized_frame, final_smoothed_count


INPUT_VIDEO_PATH = 'input1.mp4' 
OUTPUT_VIDEO_PATH = 'output_crowd_monitoring.avi' 

if not os.path.exists(INPUT_VIDEO_PATH):
    print(f"Error: Input video file not found at '{INPUT_VIDEO_PATH}'. Please ensure the path is correct.")
else:
    cap = cv2.VideoCapture(INPUT_VIDEO_PATH)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'MJPG') 
    out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
    
    if not out.isOpened():
        print(f"CRITICAL ERROR: cv2.VideoWriter failed to initialize with FOURCC: 'MJPG' at path: {OUTPUT_VIDEO_PATH}")
        print("This usually means the required codec (e.g., FFMPEG, MJPG) is missing or cannot be accessed.")
        cap.release()
        exit()


    frame_count = 0
    start_time = time.time()

    print(f"Starting video processing: {frame_total} frames at {fps} FPS.")
    print(f"Output will be saved as {OUTPUT_VIDEO_PATH} (MJPG/AVI format).")

    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        processed_frame, final_count = process_frame_pipeline(
            frame, 
            person_detector, 
            face_cascade, 
            csr_net, 
            csr_transform, 
            device,
            CSRNet_LOADED_SUCCESSFULLY 
        )
        
        out.write(processed_frame)
        
        frame_count += 1

        if frame_count % 50 == 0:
            elapsed_time = time.time() - start_time
            print(f"Progress: {frame_count}/{frame_total} frames. Final Smoothed Count: {final_count}. FPS: {frame_count / elapsed_time:.2f}")
            
    cap.release()
    out.release()
    cv2.destroyAllWindows()

    total_time = time.time() - start_time
    print(f"\nProcessing finished successfully!")
    
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Output saved to: {OUTPUT_VIDEO_PATH}")